In [15]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '/var/www/python/Qingcheng/Darwin')
sys.path.append('/var/www/python/Prod/nighthawk/')
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from nighthawk.util import bigquery_functions, connections
from nighthawk.models.valuation import node_price_predictor
import time
import pickle
import dill
from google.cloud import bigquery, storage
warnings.filterwarnings("ignore")
from datetime import datetime
from nighthawk.models.valuation import ve_model_functions
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import sys, os, inspect, shutil, importlib
sys.path.insert(0, '/var/www/python/Qingcheng/Fourier')
import pandas as pd
import numpy as np
from nighthawk.util import bigquery_functions, connections, sql_functions, dataframe_functions
from nighthawk.data.pipeline.var_handler import loadwindgen_vh, wind_vh
from nighthawk.data.pipeline.common_functions import wind
from google.cloud import bigquery
import utils_fourier.ve_portfolio_constructor_fourier as ve_portfolio_constructor_fourier
from common_functions import get_scale_factor
from nighthawk.data.network.node import Node
sys.path.insert(0, '/var/www/python/Qingcheng/QCTest/Ultra')
from custom import get_metrics, eval_valuation_model, run_in_kubernetes, fourier_port, simulate_total_ftp, select_unique_nodes_across_dates, pnl_metrics, run_valuation_backtest
import math


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
def NN_training_module_shuffle(node_num,dt):
    run_number    = 1
    opexchange    = 'SPP'

    if int(run_number) == 1:
        data_location = 'training'
    else:
        data_location = 'secondRun'

    # node_num = 1005
    # dt = '2026-01-22'

    opexchange     = "SPP"
    data_location  = "training"  # 1=training, 2=secondRun
    gs_loc         = f"gs://ve_fourier/production/SPP/{data_location}"

    max_retries = 3
    retry_delay = 15
    for attempt in range(max_retries):
        try:
            data_df = pd.read_csv(f"{gs_loc}/{node_num}_{dt}.csv")
            data_df["dt"] = pd.to_datetime(data_df["dt"]).dt.strftime("%Y-%m-%d")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                raise

    selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]

    feb2021_dates = pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"), columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist()
    if dt not in feb2021_dates:
        data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))


    class DNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size):
            super(DNN, self).__init__()
            self.fc1 = nn.Linear(input_size, hidden_size_1)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.fc3 = nn.Linear(hidden_size_2, output_size)
        def forward(self, x):
            out = self.relu(self.fc1(x))
            out = self.relu2(self.fc2(out))
            return self.fc3(out)


    class QuantileDNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size, quantiles):
            super(QuantileDNN, self).__init__()
            self.quantiles = quantiles
            self.output_size = output_size
            self.fc1 = nn.Linear(input_size, hidden_size_1)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.fc3 = nn.Linear(hidden_size_2, output_size * len(quantiles))
        def forward(self, x):
            out = self.relu(self.fc1(x))
            out = self.relu2(self.fc2(out))
            x = self.fc3(out)
            return x.view(x.size(0), self.output_size, len(self.quantiles))


    class QuantileLoss(nn.Module):
        def __init__(self, quantiles):
            super(QuantileLoss, self).__init__()
            self.quantiles = quantiles
        def forward(self, predictions, targets):
            losses = []
            for i, q in enumerate(self.quantiles):
                errors = targets - predictions[:, :, i]
                losses.append(torch.max((q - 1) * errors, q * errors).unsqueeze(2))
            return torch.mean(torch.cat(losses, dim=2))


    def predict_and_collect(model, data_loader, criterion, scaler_y=None):
        model.eval()
        total_loss = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            if predictions.ndim == 3:
                num_samples, num_outputs, num_quantiles = predictions.shape
                transposed = np.transpose(predictions, (0, 2, 1)).reshape(-1, num_outputs)
                transposed = scaler_y.inverse_transform(transposed)
                predictions = np.transpose(transposed.reshape(num_samples, num_quantiles, num_outputs), (0, 2, 1))
            else:
                predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)


    quantiles      = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99]
    quantile_names = [f"q{int(q * 100)}" for q in quantiles]
    hidden_size_1  = 32
    hidden_size_2  = 8
    learning_rate  = 0.0001
    num_epochs     = 200
    normalize      = True
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mean_criterion     = nn.MSELoss()
    quantile_criterion = QuantileLoss(quantiles)

    valuation_models = pd.DataFrame()

    for y_var in [["da_total", "rt_total"]]:
        trainingPeriod = int(min((pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
        xtrain, ytrain, xtest, ytest = ve_model_functions.getTrainTestData(
            opexchange, data_df, y=y_var, bidDt=dt, hour_gap=18,
            trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")


        # 3-way split: 1/2 train, 1/4 validate, 1/4 test
        xtrain, xtest, ytrain, ytest = train_test_split(xtrain, ytrain, test_size=1/8, shuffle=False)
        xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/7, shuffle=False)
        scaler_x, scaler_y = None, None
        if normalize:
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=xtrain.columns,    index=xtrain.index)
            xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=xvalidate.columns, index=xvalidate.index)
            xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=xtest.columns,     index=xtest.index)
            ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=ytrain.columns,    index=ytrain.index)
            yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=yvalidate.columns, index=yvalidate.index)
            ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=ytest.columns,     index=ytest.index)
        X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
        Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
        X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
        Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
        X_test_tensor     = torch.tensor(xtest.values,    dtype=torch.float32).to(device)
        Y_test_tensor     = torch.tensor(ytest.values,    dtype=torch.float32).to(device)

        g = torch.Generator()
        g.manual_seed(42)
        train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True, generator =g )
        validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
        test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)
        input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]
        mean_model     = DNN(input_size, hidden_size_1, hidden_size_2, output_size).to(device)
        quantile_model = QuantileDNN(input_size, hidden_size_1, hidden_size_2, output_size, quantiles).to(device)
        mean_optimizer     = optim.Adam(mean_model.parameters(),     lr=learning_rate)
        quantile_optimizer = optim.Adam(quantile_model.parameters(), lr=learning_rate)

        for model, optimizer, criterion in [(mean_model, mean_optimizer, mean_criterion),
                                            (quantile_model, quantile_optimizer, quantile_criterion)]:
            best_val_loss, patience, no_improve = float("inf"), 5, 0
            for epoch in range(num_epochs):
                model.train()
                for inputs, labels in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(inputs), labels)
                    loss.backward()
                    optimizer.step()
                model.eval()
                val_loss = sum(criterion(model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
                if val_loss < best_val_loss:
                    best_val_loss, no_improve = val_loss, 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"Early stopping at epoch {epoch + 1}")
                        break


        mean_preds, _     = predict_and_collect(mean_model,     test_loader, mean_criterion,     scaler_y)
        quantile_preds, _ = predict_and_collect(quantile_model, test_loader, quantile_criterion, scaler_y)
        
        if normalize and scaler_y is not None:
            ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=ytest.columns, index=ytest.index)

        mean_df     = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
        q_cols      = [f"{y}_{n}" for y in y_var for n in quantile_names]
        quantile_df = pd.DataFrame(quantile_preds.reshape(-1, output_size * len(quantiles)), columns=q_cols)

        result = pd.concat([ytest.reset_index(), mean_df, quantile_df], axis=1)
        result["dtHr"] = pd.to_datetime(result["dtHr"])
        result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
        result["hr"]   = result["dtHr"].dt.hour + 1
        valuation_models = pd.concat([valuation_models, result])

    valuation_models = valuation_models.groupby(["dt", "hr", "node_num"]).max().reset_index()
    keep_cols = ["dt", "hr", "node_num", "da_total_mean", "rt_total_mean"] + \
                [f"da_total_{n}" for n in quantile_names] + [f"rt_total_{n}" for n in quantile_names]
    valuation_models["model"] = f"dnn_{hidden_size_1}h1_{hidden_size_2}h2"

    return valuation_models 

def NN_training_module_no_shuffle(node_num,dt):
    run_number    = 1
    opexchange    = 'SPP'

    if int(run_number) == 1:
        data_location = 'training'
    else:
        data_location = 'secondRun'

    # node_num = 1005
    # dt = '2026-01-22'

    opexchange     = "SPP"
    data_location  = "training"  # 1=training, 2=secondRun
    gs_loc         = f"gs://ve_fourier/production/SPP/{data_location}"

    max_retries = 3
    retry_delay = 15
    for attempt in range(max_retries):
        try:
            data_df = pd.read_csv(f"{gs_loc}/{node_num}_{dt}.csv")
            data_df["dt"] = pd.to_datetime(data_df["dt"]).dt.strftime("%Y-%m-%d")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                raise

    selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]

    feb2021_dates = pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"), columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist()
    if dt not in feb2021_dates:
        data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))


    class DNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size):
            super(DNN, self).__init__()
            self.fc1 = nn.Linear(input_size, hidden_size_1)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.fc3 = nn.Linear(hidden_size_2, output_size)
        def forward(self, x):
            out = self.relu(self.fc1(x))
            out = self.relu2(self.fc2(out))
            return self.fc3(out)


    class QuantileDNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size, quantiles):
            super(QuantileDNN, self).__init__()
            self.quantiles = quantiles
            self.output_size = output_size
            self.fc1 = nn.Linear(input_size, hidden_size_1)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.fc3 = nn.Linear(hidden_size_2, output_size * len(quantiles))
        def forward(self, x):
            out = self.relu(self.fc1(x))
            out = self.relu2(self.fc2(out))
            x = self.fc3(out)
            return x.view(x.size(0), self.output_size, len(self.quantiles))


    class QuantileLoss(nn.Module):
        def __init__(self, quantiles):
            super(QuantileLoss, self).__init__()
            self.quantiles = quantiles
        def forward(self, predictions, targets):
            losses = []
            for i, q in enumerate(self.quantiles):
                errors = targets - predictions[:, :, i]
                losses.append(torch.max((q - 1) * errors, q * errors).unsqueeze(2))
            return torch.mean(torch.cat(losses, dim=2))


    def predict_and_collect(model, data_loader, criterion, scaler_y=None):
        model.eval()
        total_loss = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            if predictions.ndim == 3:
                num_samples, num_outputs, num_quantiles = predictions.shape
                transposed = np.transpose(predictions, (0, 2, 1)).reshape(-1, num_outputs)
                transposed = scaler_y.inverse_transform(transposed)
                predictions = np.transpose(transposed.reshape(num_samples, num_quantiles, num_outputs), (0, 2, 1))
            else:
                predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)


    quantiles      = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99]
    quantile_names = [f"q{int(q * 100)}" for q in quantiles]
    hidden_size_1  = 32
    hidden_size_2  = 8
    learning_rate  = 0.0001
    num_epochs     = 200
    normalize      = True
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mean_criterion     = nn.MSELoss()
    quantile_criterion = QuantileLoss(quantiles)

    valuation_models = pd.DataFrame()

    for y_var in [["da_total", "rt_total"]]:
        trainingPeriod = int(min((pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
        xtrain, ytrain, xtest, ytest = ve_model_functions.getTrainTestData(
            opexchange, data_df, y=y_var, bidDt=dt, hour_gap=18,
            trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")

        # 3-way split: 1/2 train, 1/4 validate, 1/4 test
        xtrain, xtest, ytrain, ytest = train_test_split(xtrain, ytrain, test_size=1/8, shuffle=False)
        xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/7, shuffle=False)
        scaler_x, scaler_y = None, None
        if normalize:
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=xtrain.columns,    index=xtrain.index)
            xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=xvalidate.columns, index=xvalidate.index)
            xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=xtest.columns,     index=xtest.index)
            ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=ytrain.columns,    index=ytrain.index)
            yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=yvalidate.columns, index=yvalidate.index)
            ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=ytest.columns,     index=ytest.index)
        X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
        Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
        X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
        Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
        X_test_tensor     = torch.tensor(xtest.values,    dtype=torch.float32).to(device)
        Y_test_tensor     = torch.tensor(ytest.values,    dtype=torch.float32).to(device)
        train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=False)
        validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
        test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)
        input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]
        mean_model     = DNN(input_size, hidden_size_1, hidden_size_2, output_size).to(device)
        quantile_model = QuantileDNN(input_size, hidden_size_1, hidden_size_2, output_size, quantiles).to(device)
        mean_optimizer     = optim.Adam(mean_model.parameters(),     lr=learning_rate)
        quantile_optimizer = optim.Adam(quantile_model.parameters(), lr=learning_rate)

        for model, optimizer, criterion in [(mean_model, mean_optimizer, mean_criterion),
                                            (quantile_model, quantile_optimizer, quantile_criterion)]:
            best_val_loss, patience, no_improve = float("inf"), 5, 0
            for epoch in range(num_epochs):
                model.train()
                for inputs, labels in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(inputs), labels)
                    loss.backward()
                    optimizer.step()
                model.eval()
                val_loss = sum(criterion(model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
                if val_loss < best_val_loss:
                    best_val_loss, no_improve = val_loss, 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"Early stopping at epoch {epoch + 1}")
                        break


        mean_preds, _     = predict_and_collect(mean_model,     test_loader, mean_criterion,     scaler_y)
        quantile_preds, _ = predict_and_collect(quantile_model, test_loader, quantile_criterion, scaler_y)
        
        if normalize and scaler_y is not None:
            ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=ytest.columns, index=ytest.index)

        mean_df     = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
        q_cols      = [f"{y}_{n}" for y in y_var for n in quantile_names]
        quantile_df = pd.DataFrame(quantile_preds.reshape(-1, output_size * len(quantiles)), columns=q_cols)

        result = pd.concat([ytest.reset_index(), mean_df, quantile_df], axis=1)
        result["dtHr"] = pd.to_datetime(result["dtHr"])
        result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
        result["hr"]   = result["dtHr"].dt.hour + 1
        valuation_models = pd.concat([valuation_models, result])

    valuation_models = valuation_models.groupby(["dt", "hr", "node_num"]).max().reset_index()
    keep_cols = ["dt", "hr", "node_num", "da_total_mean", "rt_total_mean"] + \
                [f"da_total_{n}" for n in quantile_names] + [f"rt_total_{n}" for n in quantile_names]
    valuation_models["model"] = f"dnn_{hidden_size_1}h1_{hidden_size_2}h2"

    return valuation_models 

def NN_training_with_spatial_encoder(node_num, dt):

    run_number    = 1
    opexchange    = 'SPP'

    if int(run_number) == 1:
        data_location = 'training'
    else:
        data_location = 'secondRun'

    opexchange     = "SPP"
    data_location  = "training"
    gs_loc         = f"gs://ve_fourier/production/SPP/{data_location}"

    max_retries = 3
    retry_delay = 15
    for attempt in range(max_retries):
        try:
            data_df = pd.read_csv(f"{gs_loc}/{node_num}_{dt}.csv")
            data_df["dt"] = pd.to_datetime(data_df["dt"]).dt.strftime("%Y-%m-%d")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                raise

    selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]

    feb2021_dates = pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"), columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist()
    if dt not in feb2021_dates:
        data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))


    # ── model classes (all inside function) ─────────────────────

    class SpatialEncoder(nn.Module):
        def __init__(self, input_size, num_groups=10, conv_hidden=32, d_spatial=64):
            super(SpatialEncoder, self).__init__()
            self.num_groups = num_groups
            self.group_size = math.ceil(input_size / num_groups)
            self.padded_size = self.num_groups * self.group_size
            self.input_size = input_size
            self.conv1 = nn.Conv1d(self.group_size, conv_hidden, kernel_size=3, padding=1)
            self.bn1   = nn.BatchNorm1d(conv_hidden)
            self.conv2 = nn.Conv1d(conv_hidden, conv_hidden, kernel_size=3, padding=1)
            self.bn2   = nn.BatchNorm1d(conv_hidden)
            self.pool  = nn.AdaptiveAvgPool1d(1)
            self.fc    = nn.Linear(conv_hidden, d_spatial)

        def forward(self, x):
            B = x.size(0)
            if self.input_size < self.padded_size:
                x = F.pad(x, (0, self.padded_size - self.input_size))
            x = x.view(B, self.num_groups, self.group_size).permute(0, 2, 1)
            x = self.bn1(F.relu(self.conv1(x)))
            x = self.bn2(F.relu(self.conv2(x)))
            x = self.pool(x).squeeze(-1)
            return self.fc(x)


    class SpatialDNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size,
                     num_groups=10, conv_hidden=32, d_spatial=64):
            super(SpatialDNN, self).__init__()
            self.spatial = SpatialEncoder(input_size, num_groups, conv_hidden, d_spatial)
            self.fc1  = nn.Linear(d_spatial, hidden_size_1)
            self.relu = nn.ReLU()
            self.fc2  = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.fc3  = nn.Linear(hidden_size_2, output_size)

        def forward(self, x):
            x = self.spatial(x)
            x = self.relu(self.fc1(x))
            x = self.relu2(self.fc2(x))
            return self.fc3(x)


    class SpatialQuantileDNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size,
                     quantiles, num_groups=10, conv_hidden=32, d_spatial=64):
            super(SpatialQuantileDNN, self).__init__()
            self.quantiles = quantiles
            self.output_size = output_size
            self.spatial = SpatialEncoder(input_size, num_groups, conv_hidden, d_spatial)
            self.fc1  = nn.Linear(d_spatial, hidden_size_1)
            self.relu = nn.ReLU()
            self.fc2  = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.fc3  = nn.Linear(hidden_size_2, output_size * len(quantiles))

        def forward(self, x):
            x = self.spatial(x)
            x = self.relu(self.fc1(x))
            x = self.relu2(self.fc2(x))
            x = self.fc3(x)
            return x.view(x.size(0), self.output_size, len(self.quantiles))


    class QuantileLoss(nn.Module):
        def __init__(self, quantiles):
            super(QuantileLoss, self).__init__()
            self.quantiles = quantiles
        def forward(self, predictions, targets):
            losses = []
            for i, q in enumerate(self.quantiles):
                errors = targets - predictions[:, :, i]
                losses.append(torch.max((q - 1) * errors, q * errors).unsqueeze(2))
            return torch.mean(torch.cat(losses, dim=2))


    def predict_and_collect(model, data_loader, criterion, scaler_y=None):
        model.eval()
        total_loss = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            if predictions.ndim == 3:
                num_samples, num_outputs, num_quantiles = predictions.shape
                transposed = np.transpose(predictions, (0, 2, 1)).reshape(-1, num_outputs)
                transposed = scaler_y.inverse_transform(transposed)
                predictions = np.transpose(transposed.reshape(num_samples, num_quantiles, num_outputs), (0, 2, 1))
            else:
                predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)


    # ── hyper-parameters ────────────────────────────────────────
    quantiles      = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99]
    quantile_names = [f"q{int(q * 100)}" for q in quantiles]
    hidden_size_1  = 32
    hidden_size_2  = 8
    learning_rate  = 0.0001
    num_epochs     = 200
    normalize      = True
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── spatial-encoder hyper-parameters ────────────────────────
    num_groups   = 20
    conv_hidden  = 64
    d_spatial    = 64

    mean_criterion     = nn.MSELoss()
    quantile_criterion = QuantileLoss(quantiles)

    valuation_models = pd.DataFrame()

    for y_var in [["da_total", "rt_total"]]:
        trainingPeriod = int(min((pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
        xtrain, ytrain, xtest, ytest = ve_model_functions.getTrainTestData(
            opexchange, data_df, y=y_var, bidDt=dt, hour_gap=18,
            trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")

        # 3-way split: 1/2 train, 1/4 validate, 1/4 test
        xtrain, xtest, ytrain, ytest = train_test_split(xtrain, ytrain, test_size=1/8, shuffle=False)
        xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/7, shuffle=False)
        scaler_x, scaler_y = None, None
        if normalize:
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=xtrain.columns,    index=xtrain.index)
            xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=xvalidate.columns, index=xvalidate.index)
            xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=xtest.columns,     index=xtest.index)
            ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=ytrain.columns,    index=ytrain.index)
            yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=yvalidate.columns, index=yvalidate.index)
            ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=ytest.columns,     index=ytest.index)
        X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
        Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
        X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
        Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
        X_test_tensor     = torch.tensor(xtest.values,     dtype=torch.float32).to(device)
        Y_test_tensor     = torch.tensor(ytest.values,     dtype=torch.float32).to(device)

        g = torch.Generator()
        g.manual_seed(42)
        train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True, generator=g)
        validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
        test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)
        input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]

        # ── build models ────────────────────────────────────────
        mean_model = SpatialDNN(input_size, hidden_size_1, hidden_size_2, output_size,
                                num_groups=num_groups, conv_hidden=conv_hidden, d_spatial=d_spatial).to(device)
        quantile_model = SpatialQuantileDNN(input_size, hidden_size_1, hidden_size_2, output_size, quantiles,
                                            num_groups=num_groups, conv_hidden=conv_hidden, d_spatial=d_spatial).to(device)
        mean_optimizer     = optim.Adam(mean_model.parameters(),     lr=learning_rate)
        quantile_optimizer = optim.Adam(quantile_model.parameters(), lr=learning_rate)

        for model, optimizer, criterion in [(mean_model, mean_optimizer, mean_criterion),
                                            (quantile_model, quantile_optimizer, quantile_criterion)]:
            best_val_loss, patience, no_improve = float("inf"), 5, 0
            for epoch in range(num_epochs):
                model.train()
                for inputs, labels in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(inputs), labels)
                    loss.backward()
                    optimizer.step()
                model.eval()
                val_loss = sum(criterion(model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
                if val_loss < best_val_loss:
                    best_val_loss, no_improve = val_loss, 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"Early stopping at epoch {epoch + 1}")
                        break

        mean_preds, _     = predict_and_collect(mean_model,     test_loader, mean_criterion,     scaler_y)
        quantile_preds, _ = predict_and_collect(quantile_model, test_loader, quantile_criterion, scaler_y)

        if normalize and scaler_y is not None:
            ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=ytest.columns, index=ytest.index)

        mean_df     = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
        q_cols      = [f"{y}_{n}" for y in y_var for n in quantile_names]
        quantile_df = pd.DataFrame(quantile_preds.reshape(-1, output_size * len(quantiles)), columns=q_cols)

        result = pd.concat([ytest.reset_index(), mean_df, quantile_df], axis=1)
        result["dtHr"] = pd.to_datetime(result["dtHr"])
        result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
        result["hr"]   = result["dtHr"].dt.hour + 1
        valuation_models = pd.concat([valuation_models, result])

    valuation_models = valuation_models.groupby(["dt", "hr", "node_num"]).max().reset_index()
    keep_cols = ["dt", "hr", "node_num", "da_total_mean", "rt_total_mean"] + \
                [f"da_total_{n}" for n in quantile_names] + [f"rt_total_{n}" for n in quantile_names]
    valuation_models["model"] = f"spatial_cnn_{conv_hidden}conv_{d_spatial}emb_{hidden_size_1}h1_{hidden_size_2}h2"

    return valuation_models

def NN_training_with_post_cnn(node_num, dt):
    import torch.nn.functional as F

    # Architecture: Input -> FC1 -> FC2 (hidden rep) -> CNN over hidden units -> output
    # CNN treats the hidden_size_2-dim vector as a 1-D sequence,
    # capturing local structure among learned hidden features.

    class PostCNNDNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size, conv_hidden=32):
            super().__init__()
            self.fc1   = nn.Linear(input_size, hidden_size_1)
            self.relu  = nn.ReLU()
            self.fc2   = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.conv  = nn.Conv1d(1, conv_hidden, kernel_size=3, padding=1)
            self.bn    = nn.BatchNorm1d(conv_hidden)
            self.pool  = nn.AdaptiveAvgPool1d(1)
            self.fc3   = nn.Linear(conv_hidden, output_size)

        def forward(self, x):
            x = self.relu(self.fc1(x))
            x = self.relu2(self.fc2(x))        # (B, hidden_size_2)
            x = x.unsqueeze(1)                 # (B, 1, hidden_size_2)
            x = self.bn(F.relu(self.conv(x)))  # (B, conv_hidden, hidden_size_2)
            x = self.pool(x).squeeze(-1)       # (B, conv_hidden)
            return self.fc3(x)

    class PostCNNQuantileDNN(nn.Module):
        def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size, quantiles, conv_hidden=32):
            super().__init__()
            self.quantiles   = quantiles
            self.output_size = output_size
            self.fc1   = nn.Linear(input_size, hidden_size_1)
            self.relu  = nn.ReLU()
            self.fc2   = nn.Linear(hidden_size_1, hidden_size_2)
            self.relu2 = nn.ReLU()
            self.conv  = nn.Conv1d(1, conv_hidden, kernel_size=3, padding=1)
            self.bn    = nn.BatchNorm1d(conv_hidden)
            self.pool  = nn.AdaptiveAvgPool1d(1)
            self.fc3   = nn.Linear(conv_hidden, output_size * len(quantiles))

        def forward(self, x):
            x = self.relu(self.fc1(x))
            x = self.relu2(self.fc2(x))
            x = x.unsqueeze(1)
            x = self.bn(F.relu(self.conv(x)))
            x = self.pool(x).squeeze(-1)
            x = self.fc3(x)
            return x.view(x.size(0), self.output_size, len(self.quantiles))

    class QuantileLoss(nn.Module):
        def __init__(self, quantiles):
            super().__init__()
            self.quantiles = quantiles
        def forward(self, predictions, targets):
            losses = []
            for i, q in enumerate(self.quantiles):
                errors = targets - predictions[:, :, i]
                losses.append(torch.max((q - 1) * errors, q * errors).unsqueeze(2))
            return torch.mean(torch.cat(losses, dim=2))

    def predict_and_collect(model, data_loader, criterion, scaler_y=None, normalize=True):
        model.eval()
        total_loss  = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                dev = next(model.parameters()).device
                inputs, labels = inputs.to(dev), labels.to(dev)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            if predictions.ndim == 3:
                ns, no, nq = predictions.shape
                t = np.transpose(predictions, (0, 2, 1)).reshape(-1, no)
                t = scaler_y.inverse_transform(t)
                predictions = np.transpose(t.reshape(ns, nq, no), (0, 2, 1))
            else:
                predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)

    opexchange    = "SPP"
    data_location = "training"
    gs_loc        = f"gs://ve_fourier/production/SPP/{data_location}"

    max_retries = 3
    retry_delay = 15
    for attempt in range(max_retries):
        try:
            data_df = pd.read_csv(f"{gs_loc}/{node_num}_{dt}.csv")
            data_df["dt"] = pd.to_datetime(data_df["dt"]).dt.strftime("%Y-%m-%d")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                raise

    selected_columns = [col for col in data_df.columns
                        if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]

    feb2021_dates = (pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"),
                     columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist())
    if dt not in feb2021_dates:
        data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr",
                       value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))

    quantiles      = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4,
                      0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99]
    quantile_names = [f"q{int(q * 100)}" for q in quantiles]
    hidden_size_1  = 32
    hidden_size_2  = 8
    conv_hidden    = 32
    learning_rate  = 0.0001
    num_epochs     = 200
    normalize      = True
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mean_criterion     = nn.MSELoss()
    quantile_criterion = QuantileLoss(quantiles)

    valuation_models = pd.DataFrame()

    for y_var in [["da_total", "rt_total"]]:
        trainingPeriod = int(min(
            (pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
        xtrain, ytrain, xtest, ytest = ve_model_functions.getTrainTestData(
            opexchange, data_df, y=y_var, bidDt=dt, hour_gap=18,
            trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")

        xtrain, xtest,     ytrain, ytest     = train_test_split(xtrain, ytrain, test_size=1/8, shuffle=False)
        xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/7, shuffle=False)

        scaler_x, scaler_y = None, None
        if normalize:
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=xtrain.columns,    index=xtrain.index)
            xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=xvalidate.columns, index=xvalidate.index)
            xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=xtest.columns,     index=xtest.index)
            ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=ytrain.columns,    index=ytrain.index)
            yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=yvalidate.columns, index=yvalidate.index)
            ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=ytest.columns,     index=ytest.index)

        X_train_t = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
        Y_train_t = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
        X_val_t   = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
        Y_val_t   = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
        X_test_t  = torch.tensor(xtest.values,     dtype=torch.float32).to(device)
        Y_test_t  = torch.tensor(ytest.values,     dtype=torch.float32).to(device)

        g = torch.Generator(); g.manual_seed(42)
        train_loader    = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=128, shuffle=True,  generator=g)
        validate_loader = DataLoader(TensorDataset(X_val_t,   Y_val_t),   batch_size=128, shuffle=False)
        test_loader     = DataLoader(TensorDataset(X_test_t,  Y_test_t),  batch_size=128, shuffle=False)

        input_size  = X_train_t.shape[1]
        output_size = Y_train_t.shape[1]

        mean_model     = PostCNNDNN(input_size, hidden_size_1, hidden_size_2, output_size, conv_hidden).to(device)
        quantile_model = PostCNNQuantileDNN(input_size, hidden_size_1, hidden_size_2, output_size, quantiles, conv_hidden).to(device)
        mean_optimizer     = optim.Adam(mean_model.parameters(),     lr=learning_rate)
        quantile_optimizer = optim.Adam(quantile_model.parameters(), lr=learning_rate)

        for model, optimizer, criterion in [
            (mean_model,     mean_optimizer,     mean_criterion),
            (quantile_model, quantile_optimizer, quantile_criterion),
        ]:
            best_val_loss, patience, no_improve = float("inf"), 5, 0
            for epoch in range(num_epochs):
                model.train()
                for inputs, labels in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(inputs), labels)
                    loss.backward()
                    optimizer.step()
                model.eval()
                val_loss = sum(criterion(model(inp.to(device)), lbl.to(device)).item()
                               for inp, lbl in validate_loader) / len(validate_loader)
                if val_loss < best_val_loss:
                    best_val_loss, no_improve = val_loss, 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"Early stopping at epoch {epoch + 1}")
                        break

        mean_preds,     _ = predict_and_collect(mean_model,     test_loader, mean_criterion,     scaler_y, normalize)
        quantile_preds, _ = predict_and_collect(quantile_model, test_loader, quantile_criterion, scaler_y, normalize)

        if normalize and scaler_y is not None:
            ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=ytest.columns, index=ytest.index)

        mean_df     = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
        q_cols      = [f"{y}_{n}" for y in y_var for n in quantile_names]
        quantile_df = pd.DataFrame(quantile_preds.reshape(-1, output_size * len(quantiles)), columns=q_cols)

        result = pd.concat([ytest.reset_index(), mean_df, quantile_df], axis=1)
        result["dtHr"] = pd.to_datetime(result["dtHr"])
        result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
        result["hr"]   = result["dtHr"].dt.hour + 1
        valuation_models = pd.concat([valuation_models, result])

    valuation_models = (valuation_models
                        .groupby(["dt", "hr", "node_num"]).max().reset_index())
    valuation_models["model"] = f"post_cnn_{conv_hidden}conv_{hidden_size_1}h1_{hidden_size_2}h2"
    return valuation_models

def pure_CNN(node_num, dt):

    run_number    = 1
    opexchange    = 'SPP'

    if int(run_number) == 1:
        data_location = 'training'
    else:
        data_location = 'secondRun'

    opexchange     = "SPP"
    data_location  = "training"
    gs_loc         = f"gs://ve_fourier/production/SPP/{data_location}"

    max_retries = 3
    retry_delay = 15
    for attempt in range(max_retries):
        try:
            data_df = pd.read_csv(f"{gs_loc}/{node_num}_{dt}.csv")
            data_df["dt"] = pd.to_datetime(data_df["dt"]).dt.strftime("%Y-%m-%d")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                raise

    selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]

    feb2021_dates = pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"), columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist()
    if dt not in feb2021_dates:
        data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))


    # ── model classes (all inside function) ─────────────────────

    class PureCNN(nn.Module):
        """
        Fully-convolutional network for mean prediction.
        No nn.Linear layers — the final Conv1d with kernel_size=1 acts as the
        prediction head, followed by global average pooling.
        """
        def __init__(self, input_size, output_size, conv_hidden=64, kernel_size=3):
            super(PureCNN, self).__init__()
            self.input_size = input_size
            self.conv1 = nn.Conv1d(1, conv_hidden, kernel_size=kernel_size, padding=kernel_size // 2)
            self.bn1   = nn.BatchNorm1d(conv_hidden)
            self.conv2 = nn.Conv1d(conv_hidden, conv_hidden, kernel_size=kernel_size, padding=kernel_size // 2)
            self.bn2   = nn.BatchNorm1d(conv_hidden)
            self.conv3 = nn.Conv1d(conv_hidden, conv_hidden, kernel_size=kernel_size, padding=kernel_size // 2)
            self.bn3   = nn.BatchNorm1d(conv_hidden)
            # Prediction head: 1x1 conv maps conv_hidden channels → output_size channels
            self.pred_conv = nn.Conv1d(conv_hidden, output_size, kernel_size=1)
            self.pool      = nn.AdaptiveAvgPool1d(1)

        def forward(self, x):
            # x shape: (B, input_size)
            x = x.unsqueeze(1)                    # (B, 1, input_size)
            x = self.bn1(F.relu(self.conv1(x)))   # (B, conv_hidden, input_size)
            x = self.bn2(F.relu(self.conv2(x)))   # (B, conv_hidden, input_size)
            x = self.bn3(F.relu(self.conv3(x)))   # (B, conv_hidden, input_size)
            x = self.pred_conv(x)                 # (B, output_size, input_size)
            x = self.pool(x).squeeze(-1)          # (B, output_size)
            return x


    class PureQuantileCNN(nn.Module):
        """
        Fully-convolutional network for quantile prediction.
        Final 1x1 conv outputs output_size * num_quantiles channels, reshaped at the end.
        """
        def __init__(self, input_size, output_size, quantiles, conv_hidden=64, kernel_size=3):
            super(PureQuantileCNN, self).__init__()
            self.input_size = input_size
            self.output_size = output_size
            self.quantiles = quantiles
            self.conv1 = nn.Conv1d(1, conv_hidden, kernel_size=kernel_size, padding=kernel_size // 2)
            self.bn1   = nn.BatchNorm1d(conv_hidden)
            self.conv2 = nn.Conv1d(conv_hidden, conv_hidden, kernel_size=kernel_size, padding=kernel_size // 2)
            self.bn2   = nn.BatchNorm1d(conv_hidden)
            self.conv3 = nn.Conv1d(conv_hidden, conv_hidden, kernel_size=kernel_size, padding=kernel_size // 2)
            self.bn3   = nn.BatchNorm1d(conv_hidden)
            # Prediction head: 1x1 conv → output_size * num_quantiles channels
            self.pred_conv = nn.Conv1d(conv_hidden, output_size * len(quantiles), kernel_size=1)
            self.pool      = nn.AdaptiveAvgPool1d(1)

        def forward(self, x):
            # x shape: (B, input_size)
            x = x.unsqueeze(1)                    # (B, 1, input_size)
            x = self.bn1(F.relu(self.conv1(x)))   # (B, conv_hidden, input_size)
            x = self.bn2(F.relu(self.conv2(x)))   # (B, conv_hidden, input_size)
            x = self.bn3(F.relu(self.conv3(x)))   # (B, conv_hidden, input_size)
            x = self.pred_conv(x)                 # (B, output_size*num_q, input_size)
            x = self.pool(x).squeeze(-1)          # (B, output_size*num_q)
            return x.view(x.size(0), self.output_size, len(self.quantiles))


    class QuantileLoss(nn.Module):
        def __init__(self, quantiles):
            super(QuantileLoss, self).__init__()
            self.quantiles = quantiles
        def forward(self, predictions, targets):
            losses = []
            for i, q in enumerate(self.quantiles):
                errors = targets - predictions[:, :, i]
                losses.append(torch.max((q - 1) * errors, q * errors).unsqueeze(2))
            return torch.mean(torch.cat(losses, dim=2))


    def predict_and_collect(model, data_loader, criterion, scaler_y=None):
        model.eval()
        total_loss = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            if predictions.ndim == 3:
                num_samples, num_outputs, num_quantiles = predictions.shape
                transposed = np.transpose(predictions, (0, 2, 1)).reshape(-1, num_outputs)
                transposed = scaler_y.inverse_transform(transposed)
                predictions = np.transpose(transposed.reshape(num_samples, num_quantiles, num_outputs), (0, 2, 1))
            else:
                predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)


    # ── hyper-parameters ────────────────────────────────────────
    quantiles      = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99]
    quantile_names = [f"q{int(q * 100)}" for q in quantiles]
    learning_rate  = 0.0001
    num_epochs     = 200
    normalize      = True
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── Pure CNN hyper-parameters ───────────────────────────────
    conv_hidden  = 64
    kernel_size  = 3

    mean_criterion     = nn.MSELoss()
    quantile_criterion = QuantileLoss(quantiles)

    valuation_models = pd.DataFrame()

    for y_var in [["da_total", "rt_total"]]:
        trainingPeriod = int(min((pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
        xtrain, ytrain, xtest, ytest = ve_model_functions.getTrainTestData(
            opexchange, data_df, y=y_var, bidDt=dt, hour_gap=18,
            trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")

        # 3-way split: 1/2 train, 1/4 validate, 1/4 test
        xtrain, xtest, ytrain, ytest = train_test_split(xtrain, ytrain, test_size=1/8, shuffle=False)
        xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/7, shuffle=False)
        scaler_x, scaler_y = None, None
        if normalize:
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=xtrain.columns,    index=xtrain.index)
            xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=xvalidate.columns, index=xvalidate.index)
            xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=xtest.columns,     index=xtest.index)
            ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=ytrain.columns,    index=ytrain.index)
            yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=yvalidate.columns, index=yvalidate.index)
            ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=ytest.columns,     index=ytest.index)
        X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
        Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
        X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
        Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
        X_test_tensor     = torch.tensor(xtest.values,     dtype=torch.float32).to(device)
        Y_test_tensor     = torch.tensor(ytest.values,     dtype=torch.float32).to(device)

        g = torch.Generator()
        g.manual_seed(42)
        train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True, generator=g)
        validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
        test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)
        input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]

        # ── build models ────────────────────────────────────────
        mean_model     = PureCNN(input_size, output_size,
                                 conv_hidden=conv_hidden, kernel_size=kernel_size).to(device)
        quantile_model = PureQuantileCNN(input_size, output_size, quantiles,
                                         conv_hidden=conv_hidden, kernel_size=kernel_size).to(device)
        mean_optimizer     = optim.Adam(mean_model.parameters(),     lr=learning_rate)
        quantile_optimizer = optim.Adam(quantile_model.parameters(), lr=learning_rate)

        for model, optimizer, criterion in [(mean_model, mean_optimizer, mean_criterion),
                                            (quantile_model, quantile_optimizer, quantile_criterion)]:
            best_val_loss, patience, no_improve = float("inf"), 5, 0
            for epoch in range(num_epochs):
                model.train()
                for inputs, labels in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(inputs), labels)
                    loss.backward()
                    optimizer.step()
                model.eval()
                val_loss = sum(criterion(model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
                if val_loss < best_val_loss:
                    best_val_loss, no_improve = val_loss, 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"Early stopping at epoch {epoch + 1}")
                        break

        mean_preds, _     = predict_and_collect(mean_model,     test_loader, mean_criterion,     scaler_y)
        quantile_preds, _ = predict_and_collect(quantile_model, test_loader, quantile_criterion, scaler_y)

        if normalize and scaler_y is not None:
            ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=ytest.columns, index=ytest.index)

        mean_df     = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
        q_cols      = [f"{y}_{n}" for y in y_var for n in quantile_names]
        quantile_df = pd.DataFrame(quantile_preds.reshape(-1, output_size * len(quantiles)), columns=q_cols)

        result = pd.concat([ytest.reset_index(), mean_df, quantile_df], axis=1)
        result["dtHr"] = pd.to_datetime(result["dtHr"])
        result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
        result["hr"]   = result["dtHr"].dt.hour + 1
        valuation_models = pd.concat([valuation_models, result])

    valuation_models = valuation_models.groupby(["dt", "hr", "node_num"]).max().reset_index()
    keep_cols = ["dt", "hr", "node_num", "da_total_mean", "rt_total_mean"] + \
                [f"da_total_{n}" for n in quantile_names] + [f"rt_total_{n}" for n in quantile_names]
    valuation_models["model"] = f"pure_cnn_{conv_hidden}conv_k{kernel_size}"

    return valuation_models

def GRU_framework(node_num, dt):

    run_number    = 1
    opexchange    = 'SPP'

    if int(run_number) == 1:
        data_location = 'training'
    else:
        data_location = 'secondRun'

    opexchange     = "SPP"
    data_location  = "training"
    gs_loc         = f"gs://ve_fourier/production/SPP/{data_location}"

    max_retries = 3
    retry_delay = 15
    for attempt in range(max_retries):
        try:
            data_df = pd.read_csv(f"{gs_loc}/{node_num}_{dt}.csv")
            data_df["dt"] = pd.to_datetime(data_df["dt"]).dt.strftime("%Y-%m-%d")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                raise

    selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]

    feb2021_dates = pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"), columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist()
    if dt not in feb2021_dates:
        data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))


    # ── model classes (all inside function) ─────────────────────

    class GRUMean(nn.Module):
        """
        GRU-based network for mean prediction.
        Treats the flat feature vector as a length-input_size sequence with 1 feature per step,
        runs a bidirectional multi-layer GRU, then projects the final hidden state to outputs.
        """
        def __init__(self, input_size, output_size, hidden_size=64, num_layers=2,
                     bidirectional=True, dropout=0.1):
            super(GRUMean, self).__init__()
            self.hidden_size   = hidden_size
            self.num_layers    = num_layers
            self.bidirectional = bidirectional
            self.gru = nn.GRU(
                input_size=1,                  # 1 value per sequence step
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=bidirectional,
                dropout=dropout if num_layers > 1 else 0.0,
            )
            # If bidirectional, the GRU outputs 2 * hidden_size features per step
            out_dim = hidden_size * (2 if bidirectional else 1)
            self.fc = nn.Linear(out_dim, output_size)

        def forward(self, x):
            # x shape: (B, input_size)
            x = x.unsqueeze(-1)                   # (B, input_size, 1) — seq_len=input_size, features=1
            out, h_n = self.gru(x)                # out: (B, input_size, hidden_size * num_directions)
            # Use the last time step's output (contains both directions if bidirectional)
            last = out[:, -1, :]                  # (B, hidden_size * num_directions)
            return self.fc(last)                  # (B, output_size)


    class GRUQuantile(nn.Module):
        """GRU-based network for quantile prediction."""
        def __init__(self, input_size, output_size, quantiles, hidden_size=64, num_layers=2,
                     bidirectional=True, dropout=0.1):
            super(GRUQuantile, self).__init__()
            self.output_size   = output_size
            self.quantiles     = quantiles
            self.hidden_size   = hidden_size
            self.num_layers    = num_layers
            self.bidirectional = bidirectional
            self.gru = nn.GRU(
                input_size=1,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=bidirectional,
                dropout=dropout if num_layers > 1 else 0.0,
            )
            out_dim = hidden_size * (2 if bidirectional else 1)
            self.fc = nn.Linear(out_dim, output_size * len(quantiles))

        def forward(self, x):
            # x shape: (B, input_size)
            x = x.unsqueeze(-1)                   # (B, input_size, 1)
            out, h_n = self.gru(x)                # (B, input_size, hidden_size * num_directions)
            last = out[:, -1, :]                  # (B, hidden_size * num_directions)
            x = self.fc(last)                     # (B, output_size * num_quantiles)
            return x.view(x.size(0), self.output_size, len(self.quantiles))


    class QuantileLoss(nn.Module):
        def __init__(self, quantiles):
            super(QuantileLoss, self).__init__()
            self.quantiles = quantiles
        def forward(self, predictions, targets):
            losses = []
            for i, q in enumerate(self.quantiles):
                errors = targets - predictions[:, :, i]
                losses.append(torch.max((q - 1) * errors, q * errors).unsqueeze(2))
            return torch.mean(torch.cat(losses, dim=2))


    def predict_and_collect(model, data_loader, criterion, scaler_y=None):
        model.eval()
        total_loss = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            if predictions.ndim == 3:
                num_samples, num_outputs, num_quantiles = predictions.shape
                transposed = np.transpose(predictions, (0, 2, 1)).reshape(-1, num_outputs)
                transposed = scaler_y.inverse_transform(transposed)
                predictions = np.transpose(transposed.reshape(num_samples, num_quantiles, num_outputs), (0, 2, 1))
            else:
                predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)


    # ── hyper-parameters ────────────────────────────────────────
    quantiles      = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99]
    quantile_names = [f"q{int(q * 100)}" for q in quantiles]
    learning_rate  = 0.0001
    num_epochs     = 200
    normalize      = True
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── GRU hyper-parameters ────────────────────────────────────
    hidden_size   = 64
    num_layers    = 2
    bidirectional = True
    dropout       = 0.1

    mean_criterion     = nn.MSELoss()
    quantile_criterion = QuantileLoss(quantiles)

    valuation_models = pd.DataFrame()

    for y_var in [["da_total", "rt_total"]]:
        trainingPeriod = int(min((pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
        xtrain, ytrain, xtest, ytest = ve_model_functions.getTrainTestData(
            opexchange, data_df, y=y_var, bidDt=dt, hour_gap=18,
            trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")

        # 3-way split: 1/2 train, 1/4 validate, 1/4 test
        xtrain, xtest, ytrain, ytest = train_test_split(xtrain, ytrain, test_size=1/8, shuffle=False)
        xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/7, shuffle=False)
        scaler_x, scaler_y = None, None
        if normalize:
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=xtrain.columns,    index=xtrain.index)
            xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=xvalidate.columns, index=xvalidate.index)
            xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=xtest.columns,     index=xtest.index)
            ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=ytrain.columns,    index=ytrain.index)
            yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=yvalidate.columns, index=yvalidate.index)
            ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=ytest.columns,     index=ytest.index)
        X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
        Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
        X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
        Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
        X_test_tensor     = torch.tensor(xtest.values,     dtype=torch.float32).to(device)
        Y_test_tensor     = torch.tensor(ytest.values,     dtype=torch.float32).to(device)

        g = torch.Generator()
        g.manual_seed(42)
        train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True, generator=g)
        validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
        test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)
        input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]

        # ── build models ────────────────────────────────────────
        mean_model = GRUMean(input_size, output_size,
                             hidden_size=hidden_size, num_layers=num_layers,
                             bidirectional=bidirectional, dropout=dropout).to(device)
        quantile_model = GRUQuantile(input_size, output_size, quantiles,
                                     hidden_size=hidden_size, num_layers=num_layers,
                                     bidirectional=bidirectional, dropout=dropout).to(device)
        mean_optimizer     = optim.Adam(mean_model.parameters(),     lr=learning_rate)
        quantile_optimizer = optim.Adam(quantile_model.parameters(), lr=learning_rate)

        for model, optimizer, criterion in [(mean_model, mean_optimizer, mean_criterion),
                                            (quantile_model, quantile_optimizer, quantile_criterion)]:
            best_val_loss, patience, no_improve = float("inf"), 5, 0
            for epoch in range(num_epochs):
                model.train()
                for inputs, labels in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(inputs), labels)
                    loss.backward()
                    optimizer.step()
                model.eval()
                val_loss = sum(criterion(model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
                if val_loss < best_val_loss:
                    best_val_loss, no_improve = val_loss, 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"Early stopping at epoch {epoch + 1}")
                        break

        mean_preds, _     = predict_and_collect(mean_model,     test_loader, mean_criterion,     scaler_y)
        quantile_preds, _ = predict_and_collect(quantile_model, test_loader, quantile_criterion, scaler_y)

        if normalize and scaler_y is not None:
            ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=ytest.columns, index=ytest.index)

        mean_df     = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
        q_cols      = [f"{y}_{n}" for y in y_var for n in quantile_names]
        quantile_df = pd.DataFrame(quantile_preds.reshape(-1, output_size * len(quantiles)), columns=q_cols)

        result = pd.concat([ytest.reset_index(), mean_df, quantile_df], axis=1)
        result["dtHr"] = pd.to_datetime(result["dtHr"])
        result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
        result["hr"]   = result["dtHr"].dt.hour + 1
        valuation_models = pd.concat([valuation_models, result])

    valuation_models = valuation_models.groupby(["dt", "hr", "node_num"]).max().reset_index()
    keep_cols = ["dt", "hr", "node_num", "da_total_mean", "rt_total_mean"] + \
                [f"da_total_{n}" for n in quantile_names] + [f"rt_total_{n}" for n in quantile_names]
    direction_tag = "bi" if bidirectional else "uni"
    valuation_models["model"] = f"gru_{direction_tag}_{hidden_size}h_{num_layers}L"

    return valuation_models

In [16]:
result_with_noshuffle, valuation_result_noshuffle = run_valuation_backtest(GRU_framework,2)

  record_id 50009
We gonna upload data into cloud sql!
data upload done!


sh: 0: getcwd() failed: No such file or directory


docker: 'docker rmi' requires at least 1 argument

Usage:  docker rmi [OPTIONS] IMAGE [IMAGE...]

See 'docker rmi --help' for more information

Login Succeeded
 
Using default tag: latest
latest: Pulling from movetocloud-999/fourier/ve_2024_nn
Digest: sha256:22c06995af4e5d82cc39deb8db214c0f8193dd36eef898519bacea228b671c5a
Status: Image is up to date for us-central1-docker.pkg.dev/movetocloud-999/fourier/ve_2024_nn:latest
us-central1-docker.pkg.dev/movetocloud-999/fourier/ve_2024_nn:latest
 
 #0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 382B done
#1 WARN: ConsistentInstructionCasing: Command 'workdir' should match the case of the command majority (uppercase) (line 3)
#1 WARN: ConsistentInstructionCasing: Command 'copy' should match the case of the command majority (uppercase) (line 4)
#1 WARN: SecretsUsedInArgOrEnv: Do not use ARG or ENV instructions for sensitive data (ENV "GOOGLE_APPLICATION_CR

KeyboardInterrupt: 

In [ ]:
# shuffle with batch size 128


,Prod
da_total_ME,-2.70
da_total_MSE,3848.45
da_total_MAE,20.37
da_total_RMSE,62.04
rt_total_ME,1.59
rt_total_MSE,9429.78
rt_total_MAE,32.17
rt_total_RMSE,97.11
